In [0]:
from pyspark.sql.functions import *

Load Bronze data

In [0]:
taxi = spark.table("taxi_case.bronze.taxi_trips")
weather = spark.table("taxi_case.bronze.weather_hourly")

In [0]:
MONTH_START = "2026-01-01"
MONTH_END = "2026-05-31"

Tagging each row with its first failing check (or null = valid)

In [0]:
taxi_checked = (
    taxi
    .withColumn("rejection_reason", lit(None).cast("string"))
    .withColumn("rejection_reason", when(
        col("tpep_pickup_datetime").isNull() | col("tpep_dropoff_datetime").isNull(),
        lit("null_timestamp")
    ).otherwise(col("rejection_reason")))
    .withColumn("rejection_reason", when(
        col("rejection_reason").isNull() & (col("tpep_dropoff_datetime") < col("tpep_pickup_datetime")),
        lit("dropoff_before_pickup")
    ).otherwise(col("rejection_reason")))
    .withColumn("rejection_reason", when(
        col("rejection_reason").isNull() &
        ((col("tpep_pickup_datetime") < MONTH_START) | (col("tpep_pickup_datetime") > MONTH_END)),
        lit("pickup_outside_range")
    ).otherwise(col("rejection_reason")))
    .withColumn("rejection_reason", when(
        col("rejection_reason").isNull() & (col("trip_distance") <= 0),
        lit("non_positive_distance")
    ).otherwise(col("rejection_reason")))
    .withColumn("rejection_reason", when(
        col("rejection_reason").isNull() & (col("total_amount") < 0),
        lit("negative_total_amount")
    ).otherwise(col("rejection_reason")))
)

Dedup before routing

In [0]:
key_cols = [c for c in taxi.columns if not c.startswith("_")]
before_count = taxi_checked.count()
taxi_deduped = taxi_checked.dropDuplicates(key_cols)
after_count = taxi_deduped.count()
duplicate_count = before_count - after_count

In [0]:
print(f"Total taxi rows pre-dedup: {before_count:,}")
print(f"Duplicates removed: {duplicate_count:,}")
print(f"Valid rows: {after_count:,} ({after_count/before_count:.2%})")

Total taxi rows pre-dedup: 18,999,282
Duplicates removed: 0
Valid rows: 18,999,282 (100.00%)


Split valid vs quarantine

In [0]:
quarantine_df = taxi_deduped.filter(col("rejection_reason").isNotNull())
valid_df = taxi_deduped.filter(col("rejection_reason").isNull()).drop("rejection_reason")

In [0]:
valid_df.write.format("delta").mode("overwrite").saveAsTable("taxi_case.silver.trips_input")
quarantine_df.write.format("delta").mode("overwrite").saveAsTable("taxi_case.silver.quarantine")

In [0]:
weather_valid = weather.filter(col("weather_hour_local").isNotNull())
weather_valid.write.format("delta").mode("overwrite").saveAsTable("taxi_case.silver.weather_input")

`DQ Report`

In [0]:
valid_count = valid_df.count()
quarantine_count = quarantine_df.count()
null_passenger_count = taxi_deduped.filter(col("passenger_count").isNull()).count()

In [0]:
print(f"Total taxi rows pre-dedup: {before_count:,}")
print(f"Duplicates removed: {duplicate_count:,}")
print(f"Valid rows: {valid_count:,} ({valid_count/after_count:.2%})")
print(f"Quarantined rows: {quarantine_count:,} ({quarantine_count/after_count:.2%})")
print(f"Null passenger_count (kept, flagged only): {null_passenger_count:,}")
print("Quarantine breakdown:")
quarantine_df.groupBy("rejection_reason").count().orderBy(desc("count")).show(truncate=False)

Total taxi rows pre-dedup: 18,999,282
Duplicates removed: 0
Valid rows: 18,196,074 (95.77%)
Quarantined rows: 803,208 (4.23%)
Null passenger_count (kept, flagged only): 4,812,280
Quarantine breakdown:
+---------------------+------+
|rejection_reason     |count |
+---------------------+------+
|non_positive_distance|575255|
|pickup_outside_range |120505|
|negative_total_amount|107443|
|dropoff_before_pickup|5     |
+---------------------+------+

